### Chatbot And RAG Evaluation

Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

1. How to create test datasets
2. How to run your RAG application on those datasets
3. How to measure your application's performance using different evaluation metrics

#### Overview
A typical RAG evaluation workflow consists of three main steps:

1. Creating a dataset with questions and their expected answers
2. Running your RAG application on those questions
3. Using evaluators to measure how well your application performed, looking at factors like:
 - Answer relevance
 - Answer accuracy
 - Retrieval quality
 
For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path="config/.env")

def require_env(var_name: str) -> str:
    """Fetch an env var or fail loudly instead of writing None into os.environ."""
    value = os.getenv(var_name)
    if not value:
        raise ValueError(f"{var_name} is missing from config/.env")
    return value

# LangSmith tracing + eval
os.environ["LANGSMITH_API_KEY"] = require_env("LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"

# Model providers used later in the notebook
os.environ["GOOGLE_GENAI_API_KEY"] = require_env("GOOGLE_GENAI_API_KEY")
os.environ["GROQ_API_KEY"] = require_env("GROQ_API_KEY")

### Chatbot Evaluation

In [ ]:
from langsmith import Client

client = Client()  # picks up LANGSMITH_API_KEY from the environment

# Sanity check: confirm the key is valid and see what datasets already exist.
# NOTE: don't print any part of the key itself, even a prefix -- that
# leaks straight into the saved notebook output.
print("Connected to:", client.api_url)
print("Existing datasets:", [d.name for d in client.list_datasets()])

In [ ]:
# Create (or reuse) the dataset for the simple chatbot eval.
# Using a dedicated variable name (not the generic "dataset_name") avoids
# accidentally reusing/overwriting it with the RAG dataset name later on.
chatbot_dataset_name = "Chatbot Evaluation"

chatbot_examples = [
    {
        "inputs": {"question": "What is LangChain?"},
        "outputs": {"answer": "A framework for building LLM applications"},
    },
    {
        "inputs": {"question": "What is LangSmith?"},
        "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
    },
    {
        "inputs": {"question": "What is OpenAI?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
    {
        "inputs": {"question": "What is Google?"},
        "outputs": {"answer": "A technology company known for search"},
    },
    {
        "inputs": {"question": "What is Mistral?"},
        "outputs": {"answer": "A company that creates Large Language Models"},
    },
]

try:
    chatbot_dataset = client.read_dataset(dataset_name=chatbot_dataset_name)
    print(f"Reusing existing dataset: {chatbot_dataset.id}")
except Exception:
    chatbot_dataset = client.create_dataset(chatbot_dataset_name)
    client.create_examples(dataset_id=chatbot_dataset.id, examples=chatbot_examples)
    print(f"Dataset created: {chatbot_dataset.id}")

### Define Metrics (LLM As A Judge)

`correctness_response` grades `my_app`'s output (`outputs["response"]`) against the reference answer. It's named differently from the RAG-side correctness grader further down since the two apps return differently-shaped outputs (`response` vs `answer`) -- reusing the same function name for both was a source of confusion in the original version.

In [ ]:
from langsmith import wrappers

gemini_client = wrappers.wrap_gemini(genai_api_key=os.getenv("GOOGLE_GENAI_API_KEY"))

eval_instructions = "You are an expert professor specialized in grading students' answers to questions."

def correctness_response(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """Grades my_app's `response` output against the reference answer, using Gemini as judge."""
    user_content = f"""You are grading the following question:
    {inputs['question']}
    Here is the real answer:
    {reference_outputs['answer']}
    You are grading the following predicted answer:
    {outputs['response']}
    Respond with CORRECT or INCORRECT:
    Grade:
    """
    response = gemini_client.chat.completions.create(
        # Use an actual Gemini model id here -- "gpt-4o-mini" doesn't exist
        # on the Gemini API and this call would 404.
        model="gemini-2.5-flash-lite",
        temperature=0,
        messages=[
            {"role": "system", "content": eval_instructions},
            {"role": "user", "content": user_content},
        ],
    ).choices[0].message.content

    # Exact string equality is brittle -- models often add punctuation/
    # whitespace/extra words. Check containment on the normalized text instead.
    return "CORRECT" in response.strip().upper() and "INCORRECT" not in response.strip().upper()

In [ ]:
## Concision - checks whether the actual output is less than 2x the length of the expected result.

def concision(outputs: dict, reference_outputs: dict) -> bool:
    return len(outputs["response"]) < 2 * len(reference_outputs["answer"])

### Run Evaluations

In [ ]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."

# NOTE: verify this model id against Google's current model list before relying
# on it -- Gemini model names/availability change often (e.g. 2.5-flash-lite is
# slated for shutdown in Oct 2026; gemini-3.1-flash-lite is the newer option).
def my_app(question: str, model: str = "google_genai:gemini-2.5-flash-lite", instructions: str = default_instructions) -> str:
    return gemini_client.chat.completions.create(
        model=model,
        temperature=0,
        messages=[
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ],
    ).choices[0].message.content

In [ ]:
### Call my_app for every datapoint
def ls_target(inputs: dict, model: str = "google_genai:gemini-2.5-flash-lite") -> dict:
    return {"response": my_app(inputs["question"], model=model)}

In [ ]:
## Run our evaluation against the default model
experiment_results = client.evaluate(
    ls_target,  ## Your AI system
    data=chatbot_dataset_name,
    evaluators=[correctness_response, concision],
    experiment_prefix="google_genai:gemini-2.5-flash-lite",
)

In [ ]:
## Run the same evaluation against a specific named model
## (reusing ls_target via functools.partial instead of redefining the function)
from functools import partial

experiment_results = client.evaluate(
    partial(ls_target, model="google_genai:gemini-2.5-flash-lite"),
    data=chatbot_dataset_name,
    evaluators=[correctness_response, concision],
    experiment_prefix="gemini-2.5-flash-lite-chatbot",
)

### Evaluation For RAG

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"  # good balance of speed/quality for English text
EMBEDDING_KWARGS = {
    "normalize_embeddings": True,  # recommended for BGE models (cosine similarity)
    "batch_size": 32,
}

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs=EMBEDDING_KWARGS,
)

In [ ]:
## RAG
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

# List of URLs to load documents from
urls = [
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

# Load documents from the URLs (a bad URL/network hiccup will raise here --
# wrap in try/except per-url if you want the pipeline to tolerate partial failures)
docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]

# Initialize a text splitter with specified chunk size and overlap
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=250, chunk_overlap=0
)

# Split the documents into chunks
doc_splits = text_splitter.split_documents(docs_list)

# Add the document chunks to the vector store
vectorstore = InMemoryVectorStore.from_documents(
    documents=doc_splits,
    embedding=embedding_model,
)

# `k` needs to go inside search_kwargs -- as_retriever(k=6) silently
# doesn't set top-k the way you'd expect.
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

In [ ]:
retriever.invoke("what is agents")

In [ ]:
from langchain.chat_models import init_chat_model

# Same model-id caveat as my_app above -- confirm this is still a valid,
# non-deprecated model id when you run this.
llm = init_chat_model("google_genai:gemini-2.5-flash-lite")
llm

In [ ]:
from langsmith import traceable

## Add decorator
@traceable()
def rag_bot(question: str) -> dict:
    ## Relevant context
    docs = retriever.invoke(question)
    # Joining with blank lines instead of a single space makes chunk
    # boundaries clear to the LLM.
    docs_string = "\n\n".join(doc.page_content for doc in docs)

    instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions. \
Use the following source documents to answer the user's questions. \
If you don't know the answer, just say that you don't know. \
Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""

    ## llm invoke
    ai_msg = llm.invoke([
        {"role": "system", "content": instructions},
        {"role": "user", "content": question},
    ])
    return {"answer": ai_msg.content, "documents": docs}

In [ ]:
rag_bot("What is agents")

### Dataset

In [ ]:
# Dedicated variable name again (rag_dataset_name, not dataset_name) so this
# never collides with chatbot_dataset_name from earlier in the notebook.
rag_dataset_name = "RAG Test Evaluation"

rag_examples = [
    {
        "inputs": {"question": "How does the ReAct agent use self-reflection? "},
        "outputs": {"answer": "ReAct integrates reasoning and acting, performing actions - such tools like Wikipedia search API - and then observing / reasoning about the tool outputs."},
    },
    {
        "inputs": {"question": "What are the types of biases that can arise with few-shot prompting?"},
        "outputs": {"answer": "The biases that can arise with few-shot prompting include (1) Majority label bias, (2) Recency bias, and (3) Common token bias."},
    },
    {
        "inputs": {"question": "What are five types of adversarial attacks?"},
        "outputs": {"answer": "Five types of adversarial attacks are (1) Token manipulation, (2) Gradient based attack, (3) Jailbreak prompting, (4) Human red-teaming, (5) Model red-teaming."},
    },
]

try:
    rag_dataset = client.read_dataset(dataset_name=rag_dataset_name)
    print(f"Reusing existing dataset: {rag_dataset.id}")
except Exception:
    rag_dataset = client.create_dataset(dataset_name=rag_dataset_name)
    client.create_examples(dataset_id=rag_dataset.id, examples=rag_examples)
    print(f"Dataset created: {rag_dataset.id}")

### Evaluators or Metrics
1. Correctness: Response vs reference answer
- Goal: Measure "how similar/correct is the RAG chain answer, relative to a ground-truth answer"
- Mode: Requires a ground truth (reference) answer supplied through a dataset
- Evaluator: Use LLM-as-judge to assess answer correctness.

In [ ]:
from typing_extensions import Annotated, TypedDict
from langchain_groq import ChatGroq

## Correctness Output Schema

# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in which the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[bool, ..., "True if the answer is correct, False otherwise."]

## correctness prompt
correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

grader_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
).with_structured_output(
    CorrectnessGrade, method="json_schema", strict=True
)

## evaluator
# Named correctness_answer (not correctness) since this grades rag_bot's
# `answer` key, while correctness_response above grades my_app's `response` key.
def correctness_answer(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    grade = grader_llm.invoke([
        {"role": "system", "content": correctness_instructions},
        {"role": "user", "content": answers},
    ])
    return grade["correct"]

### Relevance: Response vs input
The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.

In [ ]:
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
).with_structured_output(
    RelevanceGrade, method="json_schema", strict=True
)

# Evaluator
def relevance(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = relevance_llm.invoke([
        {"role": "system", "content": relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevant"]

### Groundedness: Response vs retrieved docs
Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.

In [ ]:
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[bool, ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
).with_structured_output(
    GroundedGrade, method="json_schema", strict=True
)

# Evaluator
def groundedness(inputs: dict, outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    grade = grounded_llm.invoke([
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["grounded"]

### Retrieval Relevance: Retrieved docs vs input

In [ ]:
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[bool, ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

retrieval_relevance_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
).with_structured_output(
    RetrievalRelevanceGrade, method="json_schema", strict=True
)

def retrieval_relevance(inputs: dict, outputs: dict) -> bool:
    """An evaluator for document relevance"""
    doc_string = "\n\n".join(doc.page_content for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    grade = retrieval_relevance_llm.invoke([
        {"role": "system", "content": retrieval_relevance_instructions},
        {"role": "user", "content": answer},
    ])
    return grade["relevant"]

### Run the evaluation

In [ ]:
def target(inputs: dict) -> dict:
    return rag_bot(inputs["question"])

experiment_results = client.evaluate(
    target,
    data=rag_dataset_name,  # was referencing the reassigned/ambiguous dataset_name before
    evaluators=[correctness_answer, groundedness, relevance, retrieval_relevance],
    experiment_prefix="rag-doc-relevance",
    metadata={"version": "LCEL context, gemini-2.5-flash-lite"},
)
# Explore results locally as a dataframe if you have pandas installed
experiment_results.to_pandas()